# TP9 : répliquer un cluster vers le cloud avec MirrorMaker 2

| Repère | Valeur |
|---|---|
| Moment | Jour 2, 14:15-15:10, après la démo du TP (14:10-14:15) |
| Compétences | C18 Réplication entre clusters avec MirrorMaker 2, supervision, bascule d'un consumer group ; rappel de C9 (SASL_SSL, SCRAM) et C10 (ACL) |
| Slides de rappel | S75 (architecture de MirrorMaker 2), S76 (configurer un flux), S77 (offsets et bascule), S78 (superviser) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, relancez la cellule d'outils, puis les cellules qui définissent les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le lab (démarrage d'un conteneur, création d'ACL) sans savoir ce qu'elle refait.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une **zone `# TODO`**, puis une **vérification**. Les commandes qui modifient le lab sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- Deux clusters dans ce TP : `admin = AdminClient(CONF)` vise le cluster **onprem** (`kafka-1` à `kafka-3`), `AdminClient({"bootstrap.servers": BS_CLOUD})` vise le cluster **cloud** (`kafka-cloud`). La fonction `cloud()` de la cellule de connexion fait ce second appel.
- MirrorMaker 2 n'a pas d'API Python : on le démarre et on lit ses traces par `docker compose`, et on observe ses effets sur les deux clusters avec le client Python.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
BS_CLOUD = os.environ["BS_CLOUD"]  # cluster cloud (TP9)
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, ecrites ici : elles ne font pas partie de confluent-kafka
def attendre(condition, timeout=60, pas=2, message="condition"):
    """Rappelle condition() jusqu'a obtenir une valeur vraie, et la renvoie."""
    debut = time.time()
    while True:
        try:
            valeur = condition()
        except Exception:
            valeur = None
        if valeur:
            return valeur
        if time.time() - debut >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(pas)

def tableau(lignes, entetes):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(entetes) + " |", "|" + "---|" * len(entetes)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in ligne) + " |" for ligne in lignes]
    display(Markdown("\n".join(md)))

In [ ]:
import re
import time

from confluent_kafka import Consumer, ConsumerGroupTopicPartitions, KafkaException, Producer, TopicPartition
from confluent_kafka.admin import (AclBinding, AclOperation, AclPermissionType, AlterConfigOpType, ConfigEntry,
                                   ConfigResource, OffsetSpec, ResourcePatternType, ResourceType)


# Le cluster cloud n'a qu'un noeud : un seul client suffit, gardé dans une variable
# (un AdminClient détruit avant la réponse fait échouer la requête avec _DESTROY).
CLIENT_CLOUD = AdminClient({"bootstrap.servers": BS_CLOUD, "log_level": 2})


def cloud():
    """AdminClient sur le cluster cloud (listener INTERNAL des outils, sans authentification)."""
    return CLIENT_CLOUD


def topics(client):
    """Topics d'un cluster, sans les topics internes de Kafka (__consumer_offsets...)."""
    return sorted(t for t in client.list_topics(timeout=10).topics if not t.startswith("__"))


def fin(client, topic):
    """Nombre de messages ecrits dans un topic : somme des offsets de fin. None si le topic n'existe pas."""
    meta = client.list_topics(timeout=10).topics.get(topic)
    if meta is None or meta.error is not None:
        return None
    demandes = {TopicPartition(topic, p): OffsetSpec.latest() for p in meta.partitions}
    return sum(f.result().offset for f in client.list_offsets(demandes).values())


def restants(client, topic, groupe):
    """Messages qu'un groupe n'a pas encore lus dans un topic : pour chaque partition, offset de fin moins
    offset commite (ou premier offset disponible si le groupe n'a rien commite)."""
    commites = offsets_groupe(client, groupe)
    total = 0
    for p in client.list_topics(timeout=10).topics[topic].partitions:
        tp = TopicPartition(topic, p)
        debut = client.list_offsets({tp: OffsetSpec.earliest()})[tp].result().offset
        fin_p = client.list_offsets({tp: OffsetSpec.latest()})[tp].result().offset
        total += fin_p - max(commites.get((topic, p), debut), debut)
    return total


def offsets_groupe(client, groupe):
    """{(topic, partition): offset commite} d'un consumer group."""
    futur = client.list_consumer_group_offsets([ConsumerGroupTopicPartitions(groupe)])[groupe]
    return {(tp.topic, tp.partition): tp.offset for tp in futur.result().topic_partitions if tp.offset >= 0}


def lire(bs, topic, groupe, maximum, silence=8):
    """Lit au plus `maximum` messages d'un topic avec un groupe, commite, et renvoie le nombre lu.
    S'arrete apres `silence` secondes sans message."""
    consommateur = Consumer({"bootstrap.servers": bs, "group.id": groupe, "auto.offset.reset": "earliest",
                             "enable.auto.commit": False, "log_level": 4})
    consommateur.subscribe([topic])
    lus, dernier = 0, time.time()
    while lus < maximum and time.time() - dernier < silence:
        message = consommateur.poll(1)
        if message is None or message.error():
            continue
        lus, dernier = lus + 1, time.time()
    if lus:
        consommateur.commit(asynchronous=False)
    consommateur.close()
    return lus


def etat_conteneur(nom):
    """Etat Docker d'un conteneur du lab : running, exited... ou chaine vide s'il n'existe pas."""
    sortie = !docker inspect -f '{{{{.State.Status}}}}' {nom} 2>/dev/null 2>&1
    return sortie.n.strip()


admin = AdminClient(CONF)
print("onprem :", len(admin.describe_cluster().result().nodes), "brokers -",
      "topics orders, payments, refunds :", [fin(admin, t) for t in ("orders", "payments", "refunds")])

## 1. Objectifs

À la fin du TP, vous savez :

- démarrer un second cluster et le distinguer du premier ;
- lire une configuration de MirrorMaker 2 : clusters, flux, topics, topics internes ;
- diagnostiquer un MirrorMaker 2 qui ne réplique pas, alors que son conteneur tourne ;
- donner à MirrorMaker 2 les seuls droits dont il a besoin sur le cluster cible ;
- basculer un consumer group d'un cluster à l'autre, et mesurer les messages relus ;
- mesurer le retard de réplication pendant une coupure, puis le rattrapage.

## 2. Durée

55 minutes, après 5 minutes de démonstration.

| Étape | Durée |
|---|---|
| 7.1 Démarrer le cluster cloud | 5 min |
| 7.2 Lire la configuration de MirrorMaker 2 | 5 min |
| 7.3 Lancer MirrorMaker 2 sans droits | 7 min |
| 7.4 Donner les droits, constater la réplication | 10 min |
| 7.5 Basculer un consumer group | 14 min |
| 7.6 Coupure et rattrapage | 7 min |
| 7.7 Défi : ajouter un topic, propager une configuration | 7 min |

## 3. Prérequis

- Lab en mode `extended`, sain : `scripts/validate.sh`. Lab démarré par `scripts/start.sh`.
- Topics `orders`, `payments` et `refunds` présents, issus des TP2 et TP4.
- Certificat du cluster cloud présent : `lab/security/certs/kafka-cloud.keystore.p12`. Sinon : `bash scripts/prepare.sh --no-pull`.
- 1 Go de mémoire disponible pour Docker, en plus du mode `extended`.
- Notions : slides S74 à S78 ; SASL_SSL, SCRAM et ACL du TP5.

## 4. Architecture utilisée

```
  Cluster onprem (source)                         Cluster cloud (cible, joue AWS MSK)
  kafka-1, kafka-2, kafka-3                       kafka-cloud, noeud unique
  orders, payments, refunds                       onprem.orders, onprem.payments
            |                                              ^
            |  lit (INTERNAL :19092)                       |  ecrit (SASL_SSL :9096, compte mm2)
            +-------------->  mirrormaker  ----------------+
                              MirrorMaker 2, mode dedie
                              config/mirrormaker/mm2.properties
```

| Élément | Valeur |
|---|---|
| Profil Compose | `mirror` : `kafka-cloud` et `mirrormaker` |
| Alias des clusters | `onprem` et `cloud` : `orders` devient `onprem.orders` sur le cluster cloud |
| Cluster cloud, outils | `BS_CLOUD` = `kafka-cloud:19092` (INTERNAL, sans authentification) |
| Cluster cloud, MirrorMaker 2 | `kafka-cloud:9096`, SASL_SSL et SCRAM-SHA-512, compte `mm2` (le port SASL/SCRAM de MSK) |

| Lab | Production |
|---|---|
| Cluster cible d'un seul nœud, facteur de réplication 1 | 3 brokers ou plus sur 3 zones, RF 3 (MSK : 2 ou 3 zones) |
| Intervalles de synchronisation de 5 à 10 s | Défauts : 60 s pour les offsets, 10 min pour les configurations et la liste des topics |
| Mot de passe de `mm2` en clair dans `mm2.properties` | Secret externalisé : fournisseur de configuration, gestionnaire de secrets (AWS Secrets Manager pour MSK) |
| Authentification SCRAM | MSK : SCRAM ou IAM (`aws-msk-iam-auth`) ; IAM n'est pas testé dans ce lab |
| MirrorMaker 2 lit la source sans authentification | Un compte sur la source aussi, avec ses ACL de lecture |

## 5. Contexte métier

NovaShop ouvre un cluster géré dans le cloud. Les applications d'analyse y seront déplacées en premier : elles doivent y retrouver les commandes et les paiements, en continu, sans que les applications du site soient touchées. Le jour de la bascule, l'équipe Expédition passera son application du cluster du site au cluster cloud : elle veut savoir combien de messages elle relira.

## 6. Énoncé

1. Démarrez le cluster cloud et prouvez qu'il est distinct du cluster onprem.
2. Lisez la configuration de MirrorMaker 2 : topics, sens, emplacement des topics internes.
3. Démarrez MirrorMaker 2 et prouvez qu'il ne réplique pas. Trouvez pourquoi.
4. Donnez au compte `mm2` les droits nécessaires sur le cluster cloud, et prouvez que `orders` et `payments` y sont recopiés, sans rien écrire sur le cluster onprem.
5. Basculez le groupe `expedition` du cluster onprem au cluster cloud, et mesurez les messages relus.
6. Coupez MirrorMaker 2 pendant une rafale de commandes : mesurez le retard, puis le rattrapage.
7. Défi : ajoutez `refunds` à la réplication, et vérifiez qu'une configuration de topic est propagée.

## 7. Étapes

### 7.1 Démarrer le cluster cloud

**Exemple en ligne de commande** : les paramètres clés du nœud cloud.

In [ ]:
!grep -E '^(process.roles|node.id|listeners|super.users|allow.everyone.if.no.acl.found|default.replication.factor)=' config/kafka/kafka-cloud/server.properties

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose --profile mirror up -d kafka-cloud
docker compose logs kafka-cloud | grep -E 'start-cloud|Formatting|Kafka Server started'
```

```bash
# [tools]
kafka-cluster.sh cluster-id --bootstrap-server $BS
kafka-cluster.sh cluster-id --bootstrap-server $BS_CLOUD
```

In [ ]:
# TODO 7.1 : demarrer le cluster cloud et le comparer au cluster onprem
# 1. Demarrez le noeud : `!docker compose --profile mirror up -d kafka-cloud`.
# 2. Attendez (attendre, timeout=90) que cloud().describe_cluster().result() reponde (enveloppez l'appel
#    dans une fonction qui renvoie None en cas de KafkaException).
# 3. id_onprem, id_cloud : attribut cluster_id de describe_cluster() sur admin et sur cloud().
# 4. noeuds_cloud : nombre de noeuds du cluster cloud ; topics_cloud : topics(cloud()).
id_onprem = id_cloud = None
noeuds_cloud = 0
topics_cloud = None

print("onprem :", id_onprem, "- cloud :", id_cloud, "-", noeuds_cloud, "noeud(s) -", topics_cloud)

In [ ]:
# Verification 7.1
assert id_cloud is not None and id_cloud != id_onprem, "A revoir : deux clusters distincts"
print("OK  deux clusters distincts")
assert noeuds_cloud == 1, "A revoir : le cluster cloud a un seul noeud"
print("OK  le cluster cloud a un seul noeud")
assert topics_cloud == [], "A revoir : le cluster cloud est vide"
print("OK  le cluster cloud est vide")

Le compte `mm2` existe déjà sur le cluster cloud, créé au formatage du stockage (lecture seule) :

In [ ]:
!docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS_CLOUD --describe --entity-type users

**Résultat attendu** :

- `process.roles=broker,controller`, `node.id=1`, trois listeners dont `SECURE://:9096`, `allow.everyone.if.no.acl.found=false`, `default.replication.factor=1` ;
- traces : `[start-cloud] Formatage du stockage (cluster tF2HiBQuSD6MqpTGGGBVfA)`, puis `Kafka Server started` ;
- deux identifiants de cluster différents ; un nœud ; aucun topic ;
- `SCRAM credential configs for user-principal 'mm2' are SCRAM-SHA-512=iterations=4096`.

### 7.2 Lire la configuration de MirrorMaker 2

La configuration est dans `lab/config/mirrormaker/mm2.properties`, montée en lecture seule dans le conteneur `mirrormaker`. Le format est celui d'un fichier `.properties` : `cle = valeur`. Une clé préfixée par un alias (`cloud.`) s'applique à ce cluster ; une clé préfixée par un flux (`onprem->cloud.`) s'applique à ce sens de réplication.

**Exemple en ligne de commande** :

```bash
# [hôte]
grep -Ev '^(#|$)' config/mirrormaker/mm2.properties
```

In [ ]:
# TODO 7.2 : lire mm2.properties en Python
# 1. mm2 : dictionnaire {cle: valeur} des lignes de "config/mirrormaker/mm2.properties" qui ne sont
#    ni vides ni des commentaires (# en tete) ; separez sur le premier "=" et retirez les espaces (strip).
# 2. flux : {cle: valeur} des entrees de mm2 dont la cle commence par "onprem->cloud." (cle sans ce prefixe).
# 3. Lisez flux et mm2, puis repondez en commentaire : quels topics ? dans quel sens ? sur quel cluster vont les
#    topics internes de MirrorMaker 2 (offset-syncs.topic.location) ?
mm2 = {}
flux = {}

print(flux)

In [ ]:
# Verification 7.2
assert flux.get("enabled") == "true" and flux.get("topics") == "orders, payments", "A revoir : flux onprem->cloud : orders, payments"
print("OK  flux onprem->cloud : orders, payments")
assert mm2.get("cloud->onprem.enabled") == "false", "A revoir : le sens cloud->onprem est desactive"
print("OK  le sens cloud->onprem est desactive")

**Résultat attendu** : le flux `onprem->cloud` est activé, pour `orders` et `payments`, avec tous les groupes (`.*`) ; le sens inverse est désactivé, et n'émet pas de heartbeats ; la cible est jointe en `SASL_SSL` sur `kafka-cloud:9096` ; les topics internes vont sur la cible (`offset-syncs.topic.location = target`).

### 7.3 Lancer MirrorMaker 2 sans droits

Le cluster cloud refuse tout ce qui n'est pas autorisé par une ACL (`allow.everyone.if.no.acl.found=false`), et le compte `mm2` n'en a aucune.

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose --profile mirror up -d mirrormaker
sleep 30
docker compose ps mirrormaker
docker compose logs mirrormaker | grep -E 'ERROR|AuthorizationException' | head -5
```

In [ ]:
# TODO 7.3 : demarrer MirrorMaker 2 et prouver qu'il ne replique pas
# 1. Demarrez le conteneur : `!docker compose --profile mirror up -d mirrormaker`, puis attendez 30 s (time.sleep).
# 2. etat : etat_conteneur("mirrormaker").
# 3. erreurs : lignes de `!docker compose logs mirrormaker` qui contiennent "AuthorizationException".
# 4. repliques : topics du cluster cloud qui commencent par "onprem.".
etat = ""
erreurs = []
repliques = None

print("conteneur :", etat, "- erreurs d'autorisation :", len(erreurs), "- topics recopies :", repliques)

In [ ]:
# Verification 7.3
assert etat == "running", "A revoir : le conteneur tourne"
print("OK  le conteneur tourne")
assert len(erreurs) > 0, "A revoir : MirrorMaker 2 est refuse par le cluster cloud"
print("OK  MirrorMaker 2 est refuse par le cluster cloud")
assert repliques == [], "A revoir : aucun topic recopie"
print("OK  aucun topic recopie")

**Résultat attendu** : le conteneur est `running` (`Up` dans `docker compose ps`), mais ses traces contiennent `TopicAuthorizationException: Not authorized to access topics: [mm2-offsets.onprem.internal]`, suivi de `Uncaught exception in herder work thread, exiting`. Aucun topic `onprem.*` sur le cluster cloud.

### 7.4 Donner les droits, constater la réplication

Le compte `mm2` a besoin, sur le cluster cloud, de créer et d'écrire :

| Ressource | Motif | Pour |
|---|---|---|
| Topics commençant par `onprem.` | `PREFIXED` | Les topics recopiés, `onprem.checkpoints.internal` |
| Topics commençant par `mm2-` | `PREFIXED` | Les topics internes de Connect et `mm2-offset-syncs.onprem.internal` |
| Topic `heartbeats` | `LITERAL` | Les heartbeats émis par le flux |
| Tous les groupes (`*`) | `LITERAL` | Le groupe de Connect et l'écriture des offsets traduits des groupes |

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-acls.sh --bootstrap-server $BS_CLOUD --add --allow-principal User:mm2 --operation All \
  --topic onprem. --resource-pattern-type prefixed
kafka-acls.sh --bootstrap-server $BS_CLOUD --add --allow-principal User:mm2 --operation All \
  --topic mm2- --resource-pattern-type prefixed
kafka-acls.sh --bootstrap-server $BS_CLOUD --add --allow-principal User:mm2 --operation All --topic heartbeats
kafka-acls.sh --bootstrap-server $BS_CLOUD --add --allow-principal User:mm2 --operation All --group '*'
```

```bash
# [hôte]
docker compose --profile mirror restart mirrormaker
```

In [ ]:
# TODO 7.4 : creer les 4 ACL sur le cluster cloud, redemarrer MirrorMaker 2, attendre la recopie
# 1. acls : les 4 AclBinding du tableau (principal "User:mm2", hote "*", AclOperation.ALL, AclPermissionType.ALLOW ;
#    ResourceType.TOPIC ou ResourceType.GROUP ; ResourcePatternType.PREFIXED ou LITERAL).
#    Creez-les sur le cluster cloud : cloud().create_acls(acls), puis .result() sur chaque futur.
# 2. Redemarrez MirrorMaker 2 : `!docker compose --profile mirror restart mirrormaker`.
# 3. Attendez (timeout=120, pas=5) que fin(cloud(), "onprem.orders") == fin(admin, "orders"), puis idem pour payments.
# 4. sur_source : topics(admin) qui commencent par "mm2-" ou valent "heartbeats" (doit rester vide).
acls = []
sur_source = None

print("ACL :", len(acls), "- ecrits sur la source :", sur_source)

In [ ]:
# Verification 7.4
admin = AdminClient(CONF)
for source in ("orders", "payments"):
    assert fin(cloud(), "onprem." + source) == fin(admin, source), "A revoir : onprem.%s : %s messages, comme %s" % (source, fin(cloud(), "onprem." + source), source)
    print("OK  onprem.%s : %s messages, comme %s" % (source, fin(cloud(), "onprem." + source), source))
assert sur_source == [], "A revoir : rien d'ecrit sur le cluster onprem"
print("OK  rien d'ecrit sur le cluster onprem")

Partitions et configuration d'un topic recopié (lecture seule) :

In [ ]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topic payments | head -1
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS_CLOUD --describe --topic onprem.payments | head -1

**Résultat attendu** :

- sur le cluster cloud : `heartbeats`, `mm2-configs.onprem.internal`, `mm2-offset-syncs.onprem.internal`, `mm2-offsets.onprem.internal`, `mm2-status.onprem.internal`, `onprem.checkpoints.internal`, `onprem.orders`, `onprem.payments` (et `__consumer_offsets` dès qu'un groupe y commite) ;
- autant de messages dans `onprem.orders` que dans `orders`, et dans `onprem.payments` que dans `payments`, en moins d'une minute ;
- aucun topic `mm2-*` ni `heartbeats` sur le cluster onprem ;
- `onprem.payments` a le même nombre de partitions que `payments`, avec un facteur de réplication 1 (celui du cluster cible).

### 7.5 Basculer un consumer group

L'application Expédition lit `orders` avec le groupe `expedition`. MirrorMaker 2 suit ses offsets (`groups = .*`) et, comme `sync.group.offsets.enabled = true`, écrit les offsets **traduits** dans le groupe `expedition` du cluster cloud, pour le topic `onprem.orders`. Fonctions de la cellule d'outils : `lire(bs, topic, groupe, maximum)` lit au plus `maximum` messages et commite ; `offsets_groupe(client, groupe)` donne les offsets commités ; `restants(client, topic, groupe)` compte les messages que le groupe n'a pas encore lus.

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-console-consumer.sh --bootstrap-server $BS --topic orders --group expedition \
  --from-beginning --max-messages 40 > /dev/null
sleep 20
kafka-consumer-groups.sh --bootstrap-server $BS --describe --group expedition
kafka-consumer-groups.sh --bootstrap-server $BS_CLOUD --describe --group expedition
kafka-console-consumer.sh --bootstrap-server $BS_CLOUD --topic onprem.orders --group expedition \
  --timeout-ms 10000 | wc -l
```

In [ ]:
# TODO 7.5 : basculer le groupe expedition et compter les messages relus
# 1. lus_source = lire(BS, "orders", "expedition", 40).
# 2. source : offsets_groupe(admin, "expedition") ; a_lire : restants(admin, "orders", "expedition")
#    (ce qui reste a lire pour l'application sur le cluster onprem).
# 3. Attendez (timeout=90, pas=5) que offsets_groupe(cloud(), "expedition") couvre autant de partitions que source ;
#    traduits = ce dictionnaire.
# 4. Avant d'executer la suite, ecrivez votre prevision en commentaire : combien de messages l'application va-t-elle lire
#    sur le cloud ?
# 5. lus_cloud = lire(BS_CLOUD, "onprem.orders", "expedition", 100000) ; doublons = lus_cloud - a_lire.
lus_source = 0
source = traduits = {}
a_lire = lus_cloud = doublons = None

print("lus sur onprem :", lus_source, "- restants :", a_lire, "- lus sur le cloud :", lus_cloud, "- doublons :", doublons)

In [ ]:
# Verification 7.5
assert lus_source == 40, "A revoir : 40 messages lus et commites sur le cluster onprem"
print("OK  40 messages lus et commites sur le cluster onprem")
assert all(k[0] == "onprem.orders" for k in traduits), "A revoir : offsets traduits pour onprem.orders"
print("OK  offsets traduits pour onprem.orders")
assert lus_cloud >= a_lire, "A revoir : aucune perte a la bascule (%s lus pour %s restants)" % (lus_cloud, a_lire)
print("OK  aucune perte a la bascule (%s lus pour %s restants)" % (lus_cloud, a_lire))

**Résultat attendu** : le groupe `expedition` existe sur les deux clusters, sur `orders` côté onprem et sur `onprem.orders` côté cloud. Sur certaines partitions, l'offset traduit est **inférieur** à l'offset d'origine : à la reprise, l'application lit tous les messages restants, plus des messages déjà traités. Aucune perte, des doublons.

### 7.6 Coupure et rattrapage

Que se passe-t-il si MirrorMaker 2 s'arrête pendant que le site continue à vendre ?

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose --profile mirror stop mirrormaker
```

```bash
# [tools]
kafka-producer-perf-test.sh --bootstrap-server $BS --topic orders --num-records 3000 --record-size 200 --throughput -1
kafka-get-offsets.sh --bootstrap-server $BS --topic orders | awk -F: '{s += $3} END {print s}'
kafka-get-offsets.sh --bootstrap-server $BS_CLOUD --topic onprem.orders | awk -F: '{s += $3} END {print s}'
```

In [ ]:
# TODO 7.6 : mesurer le retard pendant une coupure, puis le rattrapage
# 1. Arretez MirrorMaker 2 : `!docker compose --profile mirror stop mirrormaker`.
# 2. Envoyez 3000 messages de 200 octets dans orders avec un Producer (poll(0) apres chaque produce, flush() a la fin).
# 3. retard = fin(admin, "orders") - fin(cloud(), "onprem.orders").
# 4. Redemarrez : `!docker compose --profile mirror start mirrormaker` ; notez l'heure (time.time()),
#    attendez (timeout=120, pas=2) que le retard soit nul, et calculez rattrapage en secondes.
retard = None
rattrapage = None

print("retard pendant la coupure :", retard, "messages - rattrapage en", rattrapage, "s")

In [ ]:
# Verification 7.6
assert retard == 3000, "A revoir : retard de 3000 messages pendant la coupure"
print("OK  retard de 3000 messages pendant la coupure")
assert rattrapage is not None and rattrapage < 120, "A revoir : retard resorbe apres le redemarrage"
print("OK  retard resorbe apres le redemarrage")

**Résultat attendu** : un retard de 3000 messages exactement pendant la coupure ; aucun message perdu ; le retard se résorbe en une dizaine de secondes après le redémarrage, démarrage de la JVM compris.

### 7.7 Défi : ajouter un topic, propager une configuration

L'équipe Finance veut aussi `refunds` dans le cloud. Modifiez la ligne `onprem->cloud.topics` de `mm2.properties` pour y ajouter `refunds`, redémarrez MirrorMaker 2, et vérifiez que `onprem.refunds` contient autant de messages que `refunds`. Puis passez `retention.ms` d'`orders` à 3 jours (259200000) sur le cluster onprem, et vérifiez que la valeur arrive sur `onprem.orders`.

Aide : le fichier est accessible depuis le notebook à `"config/mirrormaker/mm2.properties"`. Faut-il une nouvelle ACL pour `refunds` ?

In [ ]:
# TODO 7.7 : refunds dans le flux, puis propagation de retention.ms
# 1. Remplacez "onprem->cloud.topics = orders, payments" par "onprem->cloud.topics = orders, payments, refunds"
#    dans le fichier, puis redemarrez MirrorMaker 2 (restart).
# 2. Attendez (timeout=120, pas=5) que fin(cloud(), "onprem.refunds") == fin(admin, "refunds").
# 3. Passez retention.ms d'orders a 259200000 (incremental_alter_configs, comme au TP2), puis attendez (timeout=90, pas=5)
#    que la configuration de onprem.orders sur le cloud (describe_configs) ait cette valeur.
retention_cloud = None

print("retention.ms sur le cloud :", retention_cloud)

In [ ]:
# Verification 7.7
admin = AdminClient(CONF)
assert fin(cloud(), "onprem.refunds") == fin(admin, "refunds"), "A revoir : refunds recopie dans onprem.refunds"
print("OK  refunds recopie dans onprem.refunds")
assert retention_cloud == "259200000", "A revoir : retention.ms propage sur onprem.orders"
print("OK  retention.ms propage sur onprem.orders")

**Résultat attendu** : `onprem.refunds` apparaît sur le cluster cloud, avec autant de messages que `refunds` ; aucune nouvelle ACL n'est nécessaire, le préfixe `onprem.` couvre le nouveau topic. `retention.ms=259200000` arrive sur `onprem.orders` en une quinzaine de secondes (intervalle de lab de 10 s).

## Bilan de l'exercice

In [ ]:
try:
    recopies = ", ".join(t for t in topics(cloud()) if t.startswith("onprem.") and not t.endswith("internal"))
except KafkaException:
    recopies = "cluster cloud injoignable"
tableau([
    ("Clusters distincts", "%s / %s" % (id_onprem, id_cloud), "7.1"),
    ("Erreurs d'autorisation sans ACL", len(erreurs), "7.3"),
    ("Topics recopies", recopies, "7.4, 7.7"),
    ("Bascule : restants, lus, doublons", "%s, %s, %s" % (a_lire, lus_cloud, doublons), "7.5"),
    ("Coupure : retard, rattrapage", "%s messages, %s s" % (retard, rattrapage), "7.6"),
], ["Controle", "Valeur", "Etape"])

## Fin du TP : arrêter le profil `mirror`

Le TP8 n'utilise pas le cluster cloud. Arrêtez-le pour libérer la mémoire ; ses données sont gardées jusqu'au prochain `scripts/reset.sh`.

In [ ]:
!docker compose --profile mirror stop mirrormaker kafka-cloud

## 11. Questions de réflexion

1. Le conteneur `mirrormaker` était `Up` en 7.3. Quelle supervision aurait détecté le problème ?
2. Pourquoi les topics recopiés s'appellent-ils `onprem.orders` et non `orders` ? Dans quel cas voudrait-on garder le même nom ?
3. Pourquoi l'application Expédition relit-elle des messages après la bascule ? Comment s'en protéger ?
4. MirrorMaker 2 est resté arrêté 2 minutes en 7.6. Qu'est-ce qui aurait été perdu s'il était resté arrêté 8 jours ?
5. Pourquoi placer les topics internes de MirrorMaker 2 sur le cluster cible ?
6. Le compte `mm2` a `All` sur tous les groupes du cluster cloud. Comment réduire ces droits ?
7. Chez vous, MirrorMaker 2 relie des clusters on-premise à AWS MSK. Qu'est-ce qui change par rapport au lab ?
8. Un script défectueux écrit des messages erronés dans `orders` sur le cluster onprem. MirrorMaker 2 les recopie-t-il ? Une réplique sur le cluster cloud remplace-t-elle une sauvegarde ?
9. Pourquoi relier deux clusters par MirrorMaker 2 plutôt qu'étendre un seul cluster sur les deux sites ?

## 12. Erreurs fréquentes

| Symptôme | Cause | Correction |
|---|---|---|
| `kafka-cloud` s'arrête au démarrage, `NoSuchFileException: .../kafka-cloud.keystore.p12` | Lab préparé avant l'ajout du cluster cloud : certificat absent | Sur l'hôte : `bash scripts/prepare.sh --no-pull`, puis redémarrer `kafka-cloud` |
| Conteneur `mirrormaker` `Up`, aucun topic recopié | MirrorMaker 2 refusé (ACL absente) ou cluster cloud injoignable : le processus reste en vie | Lire `docker compose logs mirrormaker` ; ajouter l'ACL ; `restart mirrormaker` |
| `SaslAuthenticationException: Authentication failed during authentication due to invalid credentials` | Mot de passe de `mm2` modifié d'un côté seulement | Même valeur dans `mm2.properties` et `MM2_PASSWORD` ; le compte n'est créé qu'au premier formatage du volume |
| Les offsets du groupe ne sont pas traduits sur le cloud | Un consommateur du groupe est actif sur le cloud, ou l'intervalle de synchronisation n'est pas écoulé | Arrêter le consommateur côté cloud ; attendre 10 s |
| Un topic ajouté à `topics` n'est pas recopié | MirrorMaker 2 non redémarré, ou topic absent de la source | `restart mirrormaker` ; vérifier le nom sur la source |
| `topics`, `heartbeats` ou `mm2-*` apparaissent sur le cluster onprem | Ligne `cloud->onprem.emit.heartbeats.enabled = false` retirée | La remettre ; supprimer ces topics de la source |

## 13. Solution formateur

Voir `corrige.ipynb` (et `corrige.md` pour la version en ligne de commande).

## 14. Variante avancée

1. **Moins de doublons.** Envoyez 1200 messages à clé dans `orders`, placez un nouveau groupe 100 messages avant la fin de chaque partition, puis basculez-le comme en 7.5 dans deux cas : groupe **à jour** (il lit tout sur onprem avant la bascule) et groupe **en retard** (il ne lit que 40 messages). Ajoutez ensuite `onprem->cloud.offset.lag.max = 10` à `mm2.properties`, redémarrez MirrorMaker 2 et refaites les deux bascules avec de nouveaux groupes. Comparez les doublons. Quel est le coût de ce réglage ?
2. **Lire les checkpoints.** Lisez le topic `onprem.checkpoints.internal` du cluster cloud avec le formateur `org.apache.kafka.connect.mirror.formatters.CheckpointFormatter` de `kafka-console-consumer.sh`. Retrouvez-y le groupe `expedition` : que signifient `upstreamOffset` et `downstreamOffset` ?